# Signal Chart Visualizer — nến (SQL) + chỉ báo + signal (CSV) + lệnh thật (backtest)

**Mục đích**: đối chứng TRỰC QUAN 4 nguồn dữ liệu độc lập trên cùng 1 biểu đồ
— chưa phải đánh giá hiệu suất, chỉ để mắt thường thấy ngay "đặt lệnh có
chuẩn không, chỉ báo đầu vào có đúng như kỳ vọng không".

- **Nến**: cache CSV trong `data_cache/` (lấy 1 lần từ SQL DP6 qua
  `core_python.db_connector.load_range()` trên OG8, chỉ SELECT).
- **Chỉ báo đầu vào** (MA/SMA + MACD Histogram): cache CSV riêng, tính bằng
  CHÍNH `core_python.configuration.run_strategy()` trên OG8 (dùng lại đúng
  `add_combo_indicators`/`add_ma_cross_indicators` — không tự tính lại độc
  lập để tránh lệch với logic tín hiệu thật). Overlay đường MA lên chart nến;
  MACD Histogram vẽ panel riêng bên dưới, đồng bộ kéo/zoom với chart nến.
- **Signal**: đọc lại đúng file CSV signal gốc (giống `signal_fidelity_check.ipynb`).
- **Lệnh thật**: `events.json`/`log.txt` của 1 lượt backtest đã archive —
  dùng lại `fidelity_lib.py` (logic đã tự-kiểm-chứng, không viết lại).

**2 LỚP marker tách biệt** (nến có signal thường KHÁC nến lệnh thực sự được
gọi ra broker — do `OnBarClosed` chỉ báo khi nến đã đóng, và do cơ chế
missing-bar fallback có thể đẩy việc xử lý sang tận vài nến sau):
- **Chấm nhỏ** cạnh nến (không đè lên thân nến) tại đúng `bartime` CSV = **nơi
  tín hiệu xuất hiện** — luôn vẽ, màu theo hướng (xanh dương=Buy/cam=Sell).
- **Mũi tên** tại đúng nến chứa `executed` trong log (thời điểm lệnh THỰC SỰ
  gọi ra broker) = **nơi lệnh vào thị trường** — chỉ vẽ khi có lệnh thật:
  - 🟢/🔴 đậm = thành giao dịch thật.
  - 🟡 nhạt (chỉ Combo) = đặt được nhưng cuối cùng KHÔNG thành.
- Tín hiệu **không hề đặt được lệnh nào** chỉ có đúng 1 chấm nhỏ.

**Chi tiết OHLC + chỉ báo + entry/SL/TP/trạng thái chỉ hiện khi RÊ CHUỘT vào**
— chart mặc định chỉ hiện ~200 nến gần nhất, kéo trái/phải để xem thêm. Chú
thích thu gọn 1 góc, mỗi dòng là 1 **checkbox lọc** — tắt/bật riêng từng nhóm
marker.

**Mở rộng thêm symbol/timeframe khác**: thêm 1 entry vào `RUNS` ở cell CONFIG
bên dưới (cần có sẵn: 1 file cache nến + 1 file cache chỉ báo trong
`data_cache/`, 1 lượt backtest đã archive) — không cần sửa gì khác.

**Kết quả**: 1 file HTML duy nhất `output/signal_chart_viewer.html`, chạy
offline hoàn toàn, dùng bản `lightweight-charts.js` vendor từ chính
`dp_program_v3` (DP6) trong `research/vendor/`.

In [1]:
import sys
import json
import bisect
from pathlib import Path
from collections import Counter

import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import fidelity_lib

RESEARCH_DIR = Path.cwd()
CBOTS_DIR = Path.home() / "Documents" / "cAlgo" / "Data" / "cBots"
OUTPUT_DIR = RESEARCH_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)


## CONFIG — nguồn chân lý duy nhất cho các dataset muốn xem

Mỗi entry = 1 combo (strategy, symbol, timeframe) đã có sẵn: 1 file cache nến
+ 1 file cache chỉ báo + 1 lượt backtest đã archive. `signal_csv_path` đọc
thẳng từ chính `parameters.cbotset` của lượt archive đó (không hardcode/đoán
lại). `overlays` = các cột line-overlay vẽ chồng lên chart nến (tên cột,
nhãn hiển thị, màu) — period trong nhãn lấy từ `config.yaml` bên OG8
(`strategies.combo.ma_period=20`, `strategies.ma_cross.fast_ma=13`/
`slow_ma=34`, xác nhận 2026-09-01).

In [2]:
RUNS = [
    {
        "label": "Combo — US30.cash / H4 — ReconcileExposure (2025 → nay)",
        "strategy": "combo",
        "archived_run_dir": CBOTS_DIR / "Combo" / "8403a83c-ae3a-44ea-a888-6e9b87c23741-Default"
                             / "ArchivedRuns" / "US30_H4_ReconcileExposure_20260901-0935",
        "candle_cache": RESEARCH_DIR / "data_cache" / "US30_H4_candles.csv",
        "indicator_cache": RESEARCH_DIR / "data_cache" / "US30_H4_combo_indicators.csv",
        "overlays": [("ma", "MA 20", "#f59e0b")],
    },
    {
        "label": "Combo — US30.cash / H4 — CLI ctrader-cli (RiskPercent=0.5%, mặc định, 2025 → nay)",
        "strategy": "combo",
        "archived_run_dir": CBOTS_DIR / "Combo" / "a3ee09ce-d849-4faa-9c1f-623ae6c50548"
                             / "ArchivedRuns" / "US30_H4_CLI_original_20260901-2142",
        "candle_cache": RESEARCH_DIR / "data_cache" / "US30_H4_candles.csv",
        "indicator_cache": RESEARCH_DIR / "data_cache" / "US30_H4_combo_indicators.csv",
        "overlays": [("ma", "MA 20", "#f59e0b")],
    },
    {
        "label": "MA Cross — US30.cash / M30 (2025 → nay)",
        "strategy": "ma_cross",
        "archived_run_dir": CBOTS_DIR / "MA Cross" / "a08e1adc-bff4-4dc4-8156-9993c09b0ecb-Default"
                             / "ArchivedRuns" / "US30_M30_AlwaysFallback_2025plus_20260901-0208",
        "candle_cache": RESEARCH_DIR / "data_cache" / "US30_M30_candles.csv",
        "indicator_cache": RESEARCH_DIR / "data_cache" / "US30_M30_ma_cross_indicators.csv",
        "overlays": [("fast_ma", "Fast SMA 13", "#38bdf8"), ("slow_ma", "Slow SMA 34", "#f59e0b")],
    },
    {
        "label": "Combo — HK50.cash / H2 — ReconcileExposure (2025 → nay)",
        "strategy": "combo",
        "archived_run_dir": CBOTS_DIR / "Combo" / "1bae1864-5246-4220-b6c4-e8fd3dbaae4e-Default"
                             / "ArchivedRuns" / "HK50_H2_ReconcileExposure_20260901-0945",
        "candle_cache": RESEARCH_DIR / "data_cache" / "HK50_H2_candles.csv",
        "indicator_cache": RESEARCH_DIR / "data_cache" / "HK50_H2_combo_indicators.csv",
        "overlays": [("ma", "MA 20", "#f59e0b")],
    },
    {
        "label": "MA Cross — HK50.cash / M45 (2025 → nay)",
        "strategy": "ma_cross",
        "archived_run_dir": CBOTS_DIR / "MA Cross" / "a08e1adc-bff4-4dc4-8156-9993c09b0ecb-Default"
                             / "ArchivedRuns" / "HK50_M45_20260901-0603",
        "candle_cache": RESEARCH_DIR / "data_cache" / "HK50_M45_candles.csv",
        "indicator_cache": RESEARCH_DIR / "data_cache" / "HK50_M45_ma_cross_indicators.csv",
        "overlays": [("fast_ma", "Fast SMA 13", "#38bdf8"), ("slow_ma", "Slow SMA 34", "#f59e0b")],
    },
]

for run in RUNS:
    params = json.loads((run["archived_run_dir"] / "parameters.cbotset").read_text(encoding="utf-8"))
    run["signal_csv_path"] = params["Parameters"]["SignalFilePath"]


## Hàm dùng chung — build 1 dataset (nến + chỉ báo + marker + tooltip detail) từ 1 CONFIG entry

In [3]:
MACD_UP_COLOR = "#22c55e"
MACD_DOWN_COLOR = "#ef4444"


def load_candles(cache_csv: Path) -> list[dict]:
    """Doc CSV nen da cache, doi bartime sang unix-seconds UTC (dung format
    lightweight-charts.js yeu cau, xem util/chart/server.py ben DP6)."""
    df = pd.read_csv(cache_csv)
    df["bartime"] = pd.to_datetime(df["bartime"], utc=True)
    return [
        {"time": int(row.bartime.timestamp()), "open": row.open, "high": row.high,
         "low": row.low, "close": row.close}
        for row in df.itertuples()
    ]


def load_indicators(cache_csv: Path, overlay_columns: list[tuple[str, str, str]]) -> dict:
    """Doc CSV chi bao (da tinh san boi chinh core_python.configuration.
    run_strategy() ben OG8 - KHONG tu tinh lai doc lap, xem docstring dau
    notebook). Tra ve {"overlays": [{label,color,data}], "macd": [...]}."""
    df = pd.read_csv(cache_csv)
    df["bartime"] = pd.to_datetime(df["bartime"], utc=True)

    overlays = []
    for column, label, color in overlay_columns:
        data = [
            {"time": int(row.bartime.timestamp()), "value": float(getattr(row, column))}
            for row in df.itertuples()
            if pd.notna(getattr(row, column))
        ]
        overlays.append({"label": label, "color": color, "data": data})

    macd = [
        {"time": int(row.bartime.timestamp()), "value": float(row.macd_h),
         "color": MACD_UP_COLOR if row.macd_h >= 0 else MACD_DOWN_COLOR}
        for row in df.itertuples()
        if pd.notna(row.macd_h)
    ] if "macd_h" in df.columns else []

    return {"overlays": overlays, "macd": macd}


def naive_utc_unix(ts: pd.Timestamp) -> int:
    """Doi 1 Timestamp UTC-naive (bartime/executed - xem CLAUDE.md: 'BarTime
    UTC-naive') sang unix-seconds ma KHONG di qua quy doi tz he thong.

    QUAN TRONG: '.timestamp()' tren Timestamp naive coi wall-clock la GIO DIA
    PHUONG cua may roi moi doi sang UTC (giong datetime.timestamp() chuan) -
    may nay chay Pacific Time (UTC-8), nen goi truc tiep se lech ~8 gio so
    voi UTC that. '.value' doc thang nanosecond tu cac truong nam/thang/ngay/
    gio... COI NHU la UTC, dung y muon vi du lieu nguon von da la UTC-naive.
    """
    return int(ts.value // 10**9)


def floor_to_candle(ts: pd.Timestamp, candle_times: list[int]) -> int | None:
    """Lam tron 1 timestamp bat ky (vd 'executed' - tick FTMO, khong khop luoi
    nen Capital.com/SQL dang ve) xuong dung nen gan nhat <= ts trong series
    dang hien thi, de marker luon roi dung vao 1 diem du lieu co that."""
    target = naive_utc_unix(ts)
    idx = bisect.bisect_right(candle_times, target) - 1
    return candle_times[idx] if idx >= 0 else None


def outcome_tier(row) -> str:
    """3 tang: 'full' (thanh giao dich that), 'partial' (dat duoc nhung khong
    thanh - chi Combo), 'fail' (khong dat duoc gi ca)."""
    if row["status"] != "placed":
        return "fail"
    fill_status = row.get("fill_status")
    if fill_status is not None and fill_status != "filled" and not pd.isna(fill_status):
        return "partial"
    return "full"


STATUS_TEXT_VI = {
    "placed": "Đã đặt lệnh",
    "before_test_window": "Ngoài khung test (không tính)",
    "fallback_expired_waiting": "Fallback hết hạn trước khi có tick khả dụng",
    "in_window_no_exact_bar_fallback_off": "Thiếu bar chính xác (fallback đang tắt)",
    "same_direction_skipped": "Bỏ qua: đã có exposure cùng hướng đang mở (ReconcileExistingExposure)",
    "same_direction_skipped_inferred": "Bỏ qua: đã có exposure cùng hướng đang mở (suy luận qua đối "
                                        "chiếu số lượng — log lượt này chưa có dòng riêng cho case này)",
    "⚠ IN_WINDOW_BUT_MISSING": "⚠ Bất thường: trong khung test mà log không nhắc",
}


# fill_status (chi Combo, tu get_fill_outcomes()) - dich sang tieng Viet ro
# nghia thay vi de nguyen ten bien tieng Anh trong tooltip (nguoi dung hoi
# 2026-09-01: "co hien lenh dat nhung khong khop/tu huy khong?" - CO, day la
# nhom do, chi can dich ro nghia hon).
FILL_STATUS_TEXT_VI = {
    "expired_unfilled": "hết hạn không khớp (tự huỷ sau 3 nến)",
    "still_pending_at_end": "còn treo tới cuối kỳ test (chưa huỷ/khớp)",
}

# Mau + nhan cho marker "dong lenh" (kind="exit") - xem fidelity_lib.get_exit_events().
# Mau CO Y chon khac han voi marker Entry (xanh/do dam) de khong bi nham lan
# du cung nam gan nhau tren chart - xem yeu cau nguoi dung 2026-09-01 sau khi
# audit 1 case "2 lenh Buy lien tiep": muon thay ro DAU la diem dong lenh cu.
EXIT_REASON_COLOR = {"sl": "#f43f5e", "tp": "#22d3ee", "other": "#94a3b8"}
EXIT_REASON_LABEL = {
    "sl": "🛑 DÍNH STOP LOSS", "tp": "✅ DÍNH TAKE PROFIT",
    "other": "⏹ ĐÓNG LỆNH (khác — cuối kỳ test/Position Management)",
}


def tooltip_lines(row: pd.Series, strategy: str) -> tuple[list[str], list[str]]:
    """(signal_lines, entry_lines) - HTML hien khi HOVER marker, khong phai
    text thuong truc tren chart (theo yeu cau nguoi dung 2026-09-01)."""
    is_buy = row["signal"] == 1
    label = "BUY" if is_buy else "SELL"

    signal_lines = [f"<b>SIGNAL {label}</b> @ {row['bartime']}"]
    if strategy == "combo" and pd.notna(row.get("entry")):
        signal_lines.append(f"CSV entry dự kiến: {row['entry']:.5g}")
    if pd.notna(row.get("atr")):
        signal_lines.append(f"ATR: {row['atr']:.5g}")
    if row["status"] == "rejected":
        status_text = f"Bị từ chối: {row.get('error')}"
    else:
        status_text = STATUS_TEXT_VI.get(row["status"], row["status"])
    signal_lines.append(f"Trạng thái: {status_text}")

    entry_lines = []
    tier = outcome_tier(row)
    if tier != "fail" and pd.notna(row.get("executed")):
        entry_lines.append(f"<b>VÀO LỆNH {label}</b> @ {row['executed']}")
        if strategy == "combo":
            if pd.notna(row.get("log_entry")):
                entry_lines.append(f"Entry: {float(row['log_entry']):.5g}")
            if pd.notna(row.get("sl")):
                entry_lines.append(f"SL: {float(row['sl']):.5g}")
            if pd.notna(row.get("tp")):
                entry_lines.append(f"TP: {float(row['tp']):.5g}")
        else:
            if pd.notna(row.get("sl")):
                entry_lines.append(f"SL: {float(row['sl']):.2f} pips (dự kiến)")
            if pd.notna(row.get("tp")):
                entry_lines.append(f"TP: {float(row['tp']):.2f} pips (dự kiến)")
        entry_lines.append(
            "Kết quả: khớp lệnh thành công" if tier == "full"
            else f"Kết quả: đặt được nhưng {FILL_STATUS_TEXT_VI.get(row.get('fill_status'), row.get('fill_status'))}"
        )
    return signal_lines, entry_lines


def exit_tooltip_lines(row: pd.Series) -> list[str]:
    """Tooltip cho marker 'dong lenh' (kind=exit) - doc THANG tu
    fidelity_lib.get_exit_events(), khong lien quan gi den merged CSV/log."""
    lines = [f"<b>{EXIT_REASON_LABEL.get(row['reason'], row['reason'])}</b> @ {row['time']}"]
    if pd.notna(row.get("entryTime")):
        hold = row["time"] - row["entryTime"]
        total_min = int(hold.total_seconds() // 60)
        h, m = divmod(total_min, 60)
        lines.append(f"Giữ lệnh: {h}h{m:02d}m (vào lúc {row['entryTime']})")
    if pd.notna(row.get("entryPrice")):
        lines.append(f"Entry: {row['entryPrice']:.5g}")
    if pd.notna(row.get("closePrice")):
        lines.append(f"Exit: {row['closePrice']:.5g}")
    if pd.notna(row.get("grossProfit")):
        sign = "+" if row["grossProfit"] >= 0 else ""
        pips_part = f" ({row['pips']:.1f} pips)" if pd.notna(row.get("pips")) else ""
        lines.append(f"P&L: {sign}{row['grossProfit']:.2f}{pips_part}")
    if pd.notna(row.get("balance")):
        lines.append(f"Balance sau: {row['balance']:.2f}")
    return lines


def build_markers_and_details(merged: pd.DataFrame, t_min: pd.Timestamp, t_max: pd.Timestamp,
                               candle_times: list[int], strategy: str):
    """2 LOP marker tach biet (xem thao luan nguoi dung 2026-09-01: 'nen co
    signal khac voi nen vao lenh, 2 marker do phai tach ra'):
      1) SIGNAL - cham nho, mau theo huong, CANH nen (belowBar/aboveBar theo
         dung huong lenh - KHONG dung inBar vi de bi de len than nen), dung
         tai bartime CSV, LUON ve.
      2) ENTRY - mui ten tai nen chua 'executed' (thoi diem lenh THUC SU goi
         ra broker) - CHI ve khi co lenh that duoc dat (tier full/partial).
    Marker KHONG mang text thuong truc (v4 ve text canh marker vinh vien,
    gay roi mat) - chi tiet day du dua vao `details` (keyed by unix time) de
    JS dung lam tooltip khi hover, xem render_viewer_html()."""
    markers = []
    details: dict[int, list[str]] = {}
    in_range = merged[(merged["bartime"] >= t_min) & (merged["bartime"] <= t_max)]
    for _, row in in_range.iterrows():
        tier = outcome_tier(row)
        is_buy = row["signal"] == 1
        signal_lines, entry_lines = tooltip_lines(row, strategy)

        sig_time = naive_utc_unix(row["bartime"])
        markers.append({
            "time": sig_time, "position": "belowBar" if is_buy else "aboveBar",
            "color": "#38bdf8" if is_buy else "#fb923c", "shape": "circle",
            # "kind"/"tier" khong thuoc chuan SeriesMarker cua lightweight-
            # charts - chi de JS loc bat/tat theo nhom (checkbox chu thich),
            # bi bo qua vo hai boi thu vien khi setMarkers().
            "kind": "signal",
        })
        details.setdefault(sig_time, []).extend(signal_lines)

        if tier != "fail" and pd.notna(row.get("executed")):
            entry_time = floor_to_candle(row["executed"], candle_times)
            if entry_time is not None:
                position = "belowBar" if is_buy else "aboveBar"
                shape = "arrowUp" if is_buy else "arrowDown"
                if tier == "full":
                    color = "#16a34a" if is_buy else "#dc2626"
                else:
                    color = "#86efac" if is_buy else "#fca5a5"
                markers.append({
                    "time": entry_time, "position": position, "color": color,
                    "shape": shape, "kind": "entry", "tier": tier,
                })
                details.setdefault(entry_time, []).extend(entry_lines)
    markers.sort(key=lambda m: m["time"])
    return markers, details


def build_exit_markers_and_details(exits: pd.DataFrame, t_min: pd.Timestamp, t_max: pd.Timestamp,
                                    candle_times: list[int]):
    """Marker rieng cho tung lenh DONG that su (Stop Loss Hit/Take Profit
    Hit/Position closed) - doc tu fidelity_lib.get_exit_events(), doc lap
    voi build_markers_and_details() (khong can merge qua CSV/log). Vi tri
    NGUOC voi entry (Buy: entry belowBar -> exit aboveBar, va nguoc lai) de
    tao hieu ung "ngoac" truc quan giua diem vao/ra 1 lenh. Hinh vuong +
    mau rieng (EXIT_REASON_COLOR) de khong nham voi mui ten Entry."""
    markers = []
    details: dict[int, list[str]] = {}
    if exits.empty:
        return markers, details
    in_range = exits[(exits["time"] >= t_min) & (exits["time"] <= t_max)]
    for _, row in in_range.iterrows():
        exit_time = floor_to_candle(row["time"], candle_times)
        if exit_time is None:
            continue
        is_buy = row["direction"] == "Buy"
        markers.append({
            "time": exit_time, "position": "aboveBar" if is_buy else "belowBar",
            "color": EXIT_REASON_COLOR.get(row["reason"], "#94a3b8"),
            "shape": "square", "kind": "exit",
        })
        details.setdefault(exit_time, []).extend(exit_tooltip_lines(row))
    markers.sort(key=lambda m: m["time"])
    return markers, details


def build_dataset(cfg: dict) -> dict:
    """1 CONFIG entry -> {label, candles, markers, markerDetails, overlays,
    macd, tier_counts}. Dung chung cho moi strategy/symbol/timeframe."""
    report, merged = fidelity_lib.build_fidelity_report(
        signal_csv_path=cfg["signal_csv_path"],
        archived_run_dir=str(cfg["archived_run_dir"]),
        strategy=cfg["strategy"],
    )
    candles = load_candles(cfg["candle_cache"])
    candle_times = [c["time"] for c in candles]
    # merged["bartime"] tz-naive (ke thua tu fidelity_lib, khop voi log tz-naive) ->
    # so sanh phai cung tz-naive, khong the doi voi Timestamp tz-aware truc tiep.
    t_min = pd.Timestamp(candles[0]["time"], unit="s", tz="UTC").tz_localize(None)
    t_max = pd.Timestamp(candles[-1]["time"], unit="s", tz="UTC").tz_localize(None)
    markers, details = build_markers_and_details(merged, t_min, t_max, candle_times, cfg["strategy"])

    exits = fidelity_lib.get_exit_events(str(cfg["archived_run_dir"]))
    exit_markers, exit_details = build_exit_markers_and_details(exits, t_min, t_max, candle_times)
    markers.extend(exit_markers)
    markers.sort(key=lambda m: m["time"])
    for k, v in exit_details.items():
        details.setdefault(k, []).extend(v)

    tier_counts = Counter(outcome_tier(r) for _, r in merged.iterrows() if t_min <= r["bartime"] <= t_max)
    indicators = load_indicators(cfg["indicator_cache"], cfg["overlays"])
    return {
        "label": cfg["label"], "candles": candles, "markers": markers,
        "markerDetails": details, "overlays": indicators["overlays"],
        "macd": indicators["macd"], "tier_counts": tier_counts,
        "n_exit_markers": len(exit_markers),
    }


datasets = [build_dataset(cfg) for cfg in RUNS]
for ds in datasets:
    label = ds["label"]
    n_candles = len(ds["candles"])
    n_markers = len(ds["markers"])
    n_macd = len(ds["macd"])
    overlay_labels = [o["label"] for o in ds["overlays"]]
    tiers = dict(ds["tier_counts"])
    print(f"{label}: {n_candles} nen, {n_markers} marker ({ds['n_exit_markers']} exit), "
          f"{n_macd} diem MACD, overlay={overlay_labels}, tier={tiers}")


Combo — US30.cash / H4 — ReconcileExposure (2025 → nay): 2561 nen, 540 marker (147 exit), 2573 diem MACD, overlay=['MA 20'], tier={'partial': 47, 'full': 147, 'fail': 5}
Combo — US30.cash / H4 — CLI ctrader-cli (RiskPercent=0.5%, mặc định, 2025 → nay): 2561 nen, 401 marker (88 exit), 2573 diem MACD, overlay=['MA 20'], tier={'partial': 26, 'full': 88, 'fail': 85}
MA Cross — US30.cash / M30 (2025 → nay): 20188 nen, 1245 marker (343 exit), 20188 diem MACD, overlay=['Fast SMA 13', 'Slow SMA 34'], tier={'full': 343, 'fail': 216}
Combo — HK50.cash / H2 — ReconcileExposure (2025 → nay): 5062 nen, 945 marker (245 exit), 5062 diem MACD, overlay=['MA 20'], tier={'full': 245, 'partial': 81, 'fail': 48}
MA Cross — HK50.cash / M45 (2025 → nay): 13326 nen, 916 marker (286 exit), 13326 diem MACD, overlay=['Fast SMA 13', 'Slow SMA 34'], tier={'full': 286, 'fail': 58}


## Render — 1 file HTML duy nhất, chart nến + overlay MA + panel MACD đồng bộ + dropdown + hover tooltip

In [4]:
def render_viewer_html(datasets: list[dict]) -> str:
    """1 trang HTML doc lap chua TAT CA dataset da build; dropdown doi
    series/markers/overlay/macd qua lai bang JS, khong reload trang, chay
    offline hoan toan. 2 chart (nen + MACD) dung 2 instance LightweightCharts
    rieng, dong bo qua lai bang subscribeVisibleTimeRangeChange (co try/catch
    chan crash khi 1 trong 2 series chua kip co du lieu luc dong bo lan dau -
    xem thao luan nguoi dung 2026-09-01, day la loi thuc te da gap)."""
    payload = [
        {"label": ds["label"], "candles": ds["candles"], "markers": ds["markers"],
         "markerDetails": ds["markerDetails"], "overlays": ds["overlays"], "macd": ds["macd"]}
        for ds in datasets
    ]
    return f"""<!doctype html>
<html lang="vi">
<head>
<meta charset="utf-8">
<title>Signal Chart Visualizer</title>
<style>
  :root{{color-scheme:dark;background:#0d1117;color:#dce3ea;font:14px Segoe UI,sans-serif}}
  *{{box-sizing:border-box}} body{{margin:0}}
  header{{padding:8px 18px;border-bottom:1px solid #273241;background:#111823;
          display:flex;align-items:center;gap:14px}}
  h1{{font-size:15px;margin:0;white-space:nowrap}}
  select{{background:#1c2734;color:#dce3ea;border:1px solid #344256;border-radius:4px;
          padding:4px 8px;font-size:13px}}
  #charts{{display:flex;flex-direction:column;height:calc(100vh - 46px)}}
  #candle-pane{{position:relative;flex:0 0 70%;min-height:0}}
  #macd-pane{{position:relative;flex:0 0 30%;min-height:0;border-top:1px solid #273241}}
  #candle-chart, #macd-chart{{position:absolute;inset:0}}
  #legend{{position:absolute;top:8px;right:8px;z-index:5;background:rgba(17,24,35,.9);
           border:1px solid #344256;border-radius:6px;padding:7px 10px;font-size:11px;
           line-height:1.9}}
  #legend label{{display:flex;align-items:center;gap:6px;cursor:pointer;white-space:nowrap;
                  user-select:none}}
  #legend input{{cursor:pointer;margin:0}}
  #legend .dot{{display:inline-block;width:9px;height:9px;border-radius:50%}}
  #legend .sq{{display:inline-block;width:8px;height:8px}}
  #legend .hint{{color:#9eb0c2;margin-top:2px;pointer-events:none}}
  #overlay-legend{{position:absolute;top:8px;left:10px;z-index:5;font-size:11px;
                    line-height:1.7;pointer-events:none}}
  #overlay-legend span{{margin-right:14px;padding:1px 6px;border-radius:3px;
                         background:rgba(17,24,35,.75)}}
  #macd-label{{position:absolute;top:6px;left:10px;z-index:5;font-size:11px;color:#9eb0c2;
               pointer-events:none}}
  #tooltip{{position:absolute;z-index:10;display:none;pointer-events:none;
            background:rgba(17,24,35,.95);border:1px solid #344256;border-radius:6px;
            padding:8px 10px;font-size:12px;line-height:1.6;max-width:260px;
            box-shadow:0 4px 14px rgba(0,0,0,.4)}}
  #tooltip .tt-ohlc{{color:#9eb0c2;margin-bottom:4px;border-bottom:1px solid #273241;
                      padding-bottom:4px}}
  #error-banner{{display:none;background:#7f1d1d;color:#fecaca;padding:8px 18px;
                 font-size:12px;white-space:pre-wrap;border-bottom:1px solid #991b1b}}
</style>
</head>
<body>
<div id="error-banner"></div>
<header>
  <h1>Signal Chart Visualizer</h1>
  <select id="dataset-select"></select>
</header>
<div id="charts">
  <div id="candle-pane">
    <div id="candle-chart"></div>
    <div id="overlay-legend"></div>
    <div id="legend">
      <label><input type="checkbox" id="f-signal" checked>
        <span class="dot" style="background:#38bdf8"></span><span class="dot" style="background:#fb923c"></span>
        Signal (Buy/Sell)</label>
      <label><input type="checkbox" id="f-full" checked>
        <span class="dot" style="background:#16a34a"></span><span class="dot" style="background:#dc2626"></span>
        Vào lệnh — thành công</label>
      <label><input type="checkbox" id="f-partial" checked>
        <span class="dot" style="background:#86efac"></span><span class="dot" style="background:#fca5a5"></span>
        Đặt được, không khớp/tự huỷ (Combo)</label>
      <label><input type="checkbox" id="f-exit" checked>
        <span class="sq" style="background:#f43f5e"></span><span class="sq" style="background:#22d3ee"></span><span class="sq" style="background:#94a3b8"></span>
        Đóng lệnh (SL / TP / khác)</label>
      <div class="hint">rê chuột vào nến/marker để xem chi tiết</div>
    </div>
    <div id="tooltip"></div>
  </div>
  <div id="macd-pane">
    <div id="macd-chart"></div>
    <div id="macd-label">MACD Histogram</div>
  </div>
</div>
<script src="../vendor/lightweight-charts.js"></script>
<script>
try {{
  // Bat moi loi JS trong toan bo script setup, hien ra banner do o dau trang -
  // thay vi that bai am tham (mot exception giua chung se dung CA SCRIPT lai,
  // giai thich vi sao nen co the hien nhung marker/chart phu phia sau lai
  // trong khong - xem thao luan 2026-09-01).
  const DATASETS = {json.dumps(payload)};
  const VISIBLE_BARS_DEFAULT = 200;

  const chartOpts = {{
    autoSize: true,
    layout: {{background: {{color: '#0d1117'}}, textColor: '#b8c5d1'}},
    grid: {{vertLines: {{color: '#1c2734'}}, horzLines: {{color: '#1c2734'}}}},
    rightPriceScale: {{borderColor: '#344256'}},
    timeScale: {{borderColor: '#344256', timeVisible: true}},
  }};

  const candleChart = LightweightCharts.createChart(document.getElementById('candle-chart'), chartOpts);
  const candleSeries = candleChart.addCandlestickSeries({{
    upColor: '#26a69a', downColor: '#ef5350', borderVisible: false,
    wickUpColor: '#26a69a', wickDownColor: '#ef5350',
  }});
  let overlaySeries = [];

  const macdChart = LightweightCharts.createChart(document.getElementById('macd-chart'), chartOpts);
  const macdSeries = macdChart.addHistogramSeries({{}});

  // Dong bo keo/zoom 2 chart qua lai theo TRUC THOI GIAN - co try/catch vi
  // 1 chart co the bat su kien "visible range doi" NGAY khi setData() cua no
  // chay, tai thoi diem series cua chart kia CHUA co du lieu (showDataset()
  // set nhieu series tuan tu, khong cung luc) - goi setVisibleRange luc do
  // nem loi noi bo thu vien ("Value is null"), tung lam dung CA SCRIPT giua
  // chung (nguyen nhan that da xac dinh qua banner loi 2026-09-01). Bo qua an
  // toan - lan dong bo tuong minh cuoi showDataset() se chinh lai dung sau.
  let syncing = false;
  function linkTimeScales(a, b) {{
    a.timeScale().subscribeVisibleTimeRangeChange(range => {{
      if (syncing || !range) return;
      syncing = true;
      try {{ b.timeScale().setVisibleRange(range); }} catch (e) {{ /* chart kia chua san sang */ }}
      syncing = false;
    }});
  }}
  linkTimeScales(candleChart, macdChart);
  linkTimeScales(macdChart, candleChart);

  // --- Tooltip: chi hien khi hover, ghep OHLC + chi bao (tu overlay/macd) +
  // chi tiet marker (tu markerDetails, keyed by unix time) ---
  const tooltip = document.getElementById('tooltip');
  const candlePane = document.getElementById('candle-pane');
  const overlayLegend = document.getElementById('overlay-legend');
  let candleMap = new Map();
  let markerDetails = {{}};
  let overlayMaps = [];   // [{{label, color, map: Map(time -> value)}}]
  let macdMap = new Map();

  candleChart.subscribeCrosshairMove(param => {{
    if (!param.time || !param.point) {{ tooltip.style.display = 'none'; return; }}
    const c = candleMap.get(param.time);
    if (!c) {{ tooltip.style.display = 'none'; return; }}
    let html = `<div class="tt-ohlc">O ${{c.open.toFixed(2)}} &nbsp;H ${{c.high.toFixed(2)}} &nbsp;`
             + `L ${{c.low.toFixed(2)}} &nbsp;C ${{c.close.toFixed(2)}}</div>`;
    overlayMaps.forEach(o => {{
      const v = o.map.get(param.time);
      if (v !== undefined) html += `<div>${{o.label}}: ${{v.toFixed(2)}}</div>`;
    }});
    const macdV = macdMap.get(param.time);
    if (macdV !== undefined) html += `<div>MACD Hist: ${{macdV.toFixed(2)}}</div>`;
    const extra = markerDetails[param.time];
    if (extra) html += extra.map(line => `<div>${{line}}</div>`).join('');
    tooltip.innerHTML = html;
    tooltip.style.display = 'block';
    const paneWidth = candlePane.clientWidth;
    let left = param.point.x + 16;
    if (left + 270 > paneWidth) left = param.point.x - 270;
    tooltip.style.left = Math.max(4, left) + 'px';
    tooltip.style.top = (param.point.y + 12) + 'px';
  }});

  // --- Chu thich = checkbox loc marker theo nhom (bat/tat rieng tung loai,
  // vd chi muon xem Signal + "dat duoc nhung khong khop") ---
  const filterBoxes = {{
    signal: document.getElementById('f-signal'),
    full: document.getElementById('f-full'),
    partial: document.getElementById('f-partial'),
    exit: document.getElementById('f-exit'),
  }};
  let currentMarkers = [];

  function applyMarkerFilter() {{
    const on = {{
      signal: filterBoxes.signal.checked,
      full: filterBoxes.full.checked,
      partial: filterBoxes.partial.checked,
      exit: filterBoxes.exit.checked,
    }};
    const visible = currentMarkers
      .filter(m => (m.kind === 'signal' && on.signal)
                 || (m.kind === 'entry' && m.tier === 'full' && on.full)
                 || (m.kind === 'entry' && m.tier === 'partial' && on.partial)
                 || (m.kind === 'exit' && on.exit))
      .map(({{time, position, color, shape}}) => ({{time, position, color, shape}}));
    candleSeries.setMarkers(visible);
  }}
  Object.values(filterBoxes).forEach(box => box.addEventListener('change', applyMarkerFilter));

  function showDataset(index) {{
    const ds = DATASETS[index];

    overlaySeries.forEach(s => candleChart.removeSeries(s));
    overlaySeries = [];
    overlayMaps = [];
    ds.overlays.forEach(o => {{
      const series = candleChart.addLineSeries({{color: o.color, lineWidth: 2}});
      series.setData(o.data);
      overlaySeries.push(series);
      overlayMaps.push({{label: o.label, color: o.color, map: new Map(o.data.map(p => [p.time, p.value]))}});
    }});
    overlayLegend.innerHTML = ds.overlays
      .map(o => `<span style="color:${{o.color}}">${{o.label}}</span>`).join('');

    macdSeries.setData(ds.macd);
    macdMap = new Map(ds.macd.map(p => [p.time, p.value]));

    candleSeries.setData(ds.candles);
    currentMarkers = ds.markers;
    applyMarkerFilter();
    candleMap = new Map(ds.candles.map(c => [c.time, c]));
    markerDetails = ds.markerDetails;

    // Mac dinh chi hien ~200 nen gan nhat - keo trai/phai de xem them (du
    // lieu da nam san trong bo nho, khong phai lazy-load tu server).
    const total = ds.candles.length;
    candleChart.timeScale().setVisibleLogicalRange({{
      from: Math.max(0, total - VISIBLE_BARS_DEFAULT), to: total + 2,
    }});
    const visibleRange = candleChart.timeScale().getVisibleRange();
    if (visibleRange) {{
      try {{ macdChart.timeScale().setVisibleRange(visibleRange); }} catch (e) {{ /* bo qua */ }}
    }}
  }}

  const select = document.getElementById('dataset-select');
  DATASETS.forEach((ds, i) => {{
    const opt = document.createElement('option');
    opt.value = i; opt.textContent = ds.label;
    select.appendChild(opt);
  }});
  select.addEventListener('change', () => showDataset(select.value));
  showDataset(0);

  // Phong thu: ep do lai kich thuoc that su 1 lan sau khi layout flex da on
  // dinh hoan toan (autoSize doi khi do sai kich thuoc ngay luc khoi tao neu
  // container chua co chieu cao cuoi cung tai thoi diem createChart()).
  requestAnimationFrame(() => {{
    const cEl = document.getElementById('candle-chart');
    const mEl = document.getElementById('macd-chart');
    candleChart.resize(cEl.clientWidth, cEl.clientHeight);
    macdChart.resize(mEl.clientWidth, mEl.clientHeight);
  }});
}} catch (err) {{
  const banner = document.getElementById('error-banner');
  banner.style.display = 'block';
  banner.textContent = 'Loi JS khi dung chart (bao lai dong nay cho Claude): '
    + (err && err.stack ? err.stack : String(err));
  console.error(err);
}}
</script>
</body>
</html>
"""


viewer_html = render_viewer_html(datasets)
viewer_out = OUTPUT_DIR / "signal_chart_viewer.html"
viewer_out.write_text(viewer_html, encoding="utf-8")
print(f"Da luu: {viewer_out} ({len(datasets)} dataset trong dropdown)")


Da luu: C:\Users\Administrator\Documents\cAlgo\Sources\Robots\research\output\signal_chart_viewer.html (5 dataset trong dropdown)


## Cách xem kết quả

Mở `research/output/signal_chart_viewer.html` bằng trình duyệt bất kỳ
(double-click hoặc kéo thả vào Chrome/Edge). Dropdown góc trên chuyển dataset;
chart nến (kèm overlay MA/SMA, nhãn góc trái) + panel MACD Histogram bên dưới
kéo/zoom đồng bộ; rê chuột vào nến hoặc marker để xem chi tiết OHLC/chỉ báo/
entry/SL/TP/trạng thái; mặc định chỉ hiện ~200 nến gần nhất, kéo trái/phải để
xem thêm lịch sử; tick/bỏ tick từng dòng chú thích góc trên phải để lọc riêng
từng nhóm marker. Chạy offline hoàn toàn.